# Reproduce all results

One click: **Run All**. The notebook

1. creates the virtual environment `.venv` in the repository (once; Python 3.12 recommended),
2. installs the pinned packages of `requirements.txt` into it,
3. runs `reproduce.py` (all four publications), `analysis/sensitivity.py` and the checks (`pytest`),
4. shows a summary of the comparison with the published values and the main figures.

The notebook kernel only needs the Python standard library and IPython; all analysis runs in `.venv`.
Run time: approx. 10-30 min for the first full run (`QUICK = True`: earlier publications only, < 1 min after
installation). Details: [README.md](README.md).

In [ ]:
# Settings
QUICK = False          # True: only SENSORS 2018 / EJRS 2019 / ISOEN 2022 (skips the ISOEN 2024 processing chain)
RUN_TESTS = True       # run pytest after the reproduction
BASE_PYTHON = None     # Python used to create .venv; None = the Python of this kernel

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT / 'reproduce.py').exists(), 'Start the notebook from the repository root (folder containing reproduce.py).'
VENV = ROOT / '.venv'
PY = VENV / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
ENV = {**os.environ, 'PYTHONIOENCODING': 'utf-8', 'MPLBACKEND': 'Agg'}


def run(*args):
    """Run a command in the repository root and stream its output; stop on error."""
    t0 = time.time()
    print('$', ' '.join(str(a) for a in args), flush=True)
    p = subprocess.Popen([str(a) for a in args], cwd=ROOT, env=ENV, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace')
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'command failed with exit code {p.returncode}')
    print(f'-- done in {time.time() - t0:.0f} s')

## 1. Python environment

In [ ]:
base = BASE_PYTHON or sys.executable
if not PY.exists():
    version = subprocess.run([base, '-c', 'import sys; print(sys.version_info[:2])'], capture_output=True, text=True).stdout.strip()
    print('creating .venv with', base, version)
    if version not in ('(3, 12)',):
        print('WARNING: tested with Python 3.12; the pinned packages (numpy 1.26) have no wheels for Python >= 3.13. '
              'Set BASE_PYTHON to a Python 3.12 executable if the installation fails.')
    run(base, '-m', 'venv', VENV)
else:
    print('.venv exists:', PY)
run(PY, '-m', 'pip', 'install', '--disable-pip-version-check', '-q', '-r', 'requirements.txt')
run(PY, '-c', 'import sys, numpy, scipy, pandas; print(sys.version.split()[0], "numpy", numpy.__version__, "scipy", scipy.__version__, "pandas", pandas.__version__)')

## 2. Reproduction

In [ ]:
if QUICK:
    run(PY, 'reproduce.py', '--only-earlier')
else:
    run(PY, 'reproduce.py')
    run(PY, 'analysis/sensitivity.py')

## 3. Checks

In [ ]:
if RUN_TESTS:
    run(PY, '-m', 'pytest', '-q', '-p', 'no:warnings', '--color=no') if not QUICK else run(PY, '-m', 'pytest', '-q', '-p', 'no:warnings', '--color=no', '-m', 'not slow')

## 4. Summary

In [ ]:
import csv
from collections import Counter
from IPython.display import HTML, Image, Markdown, display

def rows(path):
    with open(ROOT / path, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

FILES = [('ISOEN 2024, Table I', 'results/table1_comparison.csv'),
         ('EJRS 2019, Table 1 (vs. printed table)', 'results/ejrs2019/table1_comparison.csv'),
         ('EJRS 2019, experiment #3', 'results/ejrs2019/aiming_comparison.csv'),
         ('EJRS 2019, wind and reference array', 'results/ejrs2019/plume_comparison.csv'),
         ('ISOEN 2022', 'results/isoen2022/comparison.csv')]
summary = []
for name, path in FILES:
    if (ROOT / path).exists():
        r = rows(path)
        c = Counter(x['status'] for x in r)
        summary.append(f'| {name} | {len(r)} | ' + ', '.join(f'{k}: {v}' for k, v in sorted(c.items())) + f' | `{path}` |')
display(Markdown('| Result | Values | Status | File |\n|---|---:|---|---|\n' + '\n'.join(summary)))
display(Markdown('Differences are explained in [docs/verification.md](docs/verification.md) (notes K7, E1, E5, D1, D2).'))

In [ ]:
deviations = []
for name, path in FILES:
    if (ROOT / path).exists():
        for x in rows(path):
            if x['status'] == 'DEVIATION':
                label = ', '.join(f'{k} {x[k]}' for k in ('surface', 'distance_m', 'trial', 'method') if k in x)
                deviations.append(f"| {name} | {label + ': ' if label else ''}{x['quantity']} | {x['published']} | "
                                  f"{x.get('reproduced_rounded_as_published') or x['reproduced']} |")
display(Markdown('**Values deviating from the publications**\n\n| Publication | Quantity | Published | Reproduced |\n|---|---|---:|---:|\n'
                 + '\n'.join(deviations)))

In [ ]:
for path, title in [('results/figures/fig2a_trial5_reconstruction.png', 'ISOEN 2024 Fig. 2a: reconstruction, trial 5'),
                    ('results/figures/fig2b_trial5_ground_truth.png', 'ISOEN 2024 Fig. 2b: ground truth'),
                    ('results/figures/fig2c_trial5_rays_all.png', 'ISOEN 2024 Fig. 2c: all rays'),
                    ('results/figures/fig2d_trial5_rays_hitting_bag.png', 'ISOEN 2024 Fig. 2d: rays through the bag'),
                    ('results/ejrs2019/figures/plume_flight13_mean_variance.png', 'SENSORS 2018 Fig. 4 / EJRS 2019 Fig. 10a: flight 13'),
                    ('results/ejrs2019/figures/plume_flight87_mean_variance.png', 'EJRS 2019 Fig. 10b: flight 87'),
                    ('results/ejrs2019/figures/plume_flight90_mean_variance.png', 'EJRS 2019 Fig. 10c: flight 90')]:
    if (ROOT / path).exists():
        display(Markdown(f'**{title}**'), Image(filename=str(ROOT / path), width=600))

The committed results in `results/` are overwritten by the run; `git status` / `git diff` shows any difference.
Result tables are expected to be identical in content; figures may differ at byte level (font rendering).